In [1]:
!pip install -q gradio google-genai

In [7]:
import os
import gradio as gr
from google import genai
from google.colab import userdata


# ============================================
# 1. GEMINI API SETUP
# ============================================

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)


# ============================================
# 2. CODE REVIEW FUNCTION
# ============================================

def review_code(code_input, language):

    # Check whether code was entered
    if not code_input or not code_input.strip():
        return (
            "Please enter valid code to review.",
            "",
            "",
            ""
        )

    # Prompt sent to Gemini
    prompt = f"""
You are an expert Senior Software Engineer and Security Analyst.

Analyze the following {language} code:

```{language}
{code_input}
```

Provide a structured code review with exactly these four sections:

1. BUG DETECTED:
Identify logic errors, syntax issues, runtime risks, or potential bugs.
If none are found, state: "No critical bugs identified."

2. CODE STYLE & PEP8:
Identify formatting problems, naming issues, readability problems,
style violations, missing documentation, or other code-quality concerns.
For languages other than Python, discuss the relevant coding standards
instead of PEP8.

3. SECURITY ANALYSIS:
Identify potential security vulnerabilities such as:
- Unsafe input handling
- SQL injection
- Hardcoded credentials
- Insecure functions
- Improper validation
- Memory/resource risks
- Other relevant security concerns

If no significant security issues are found, clearly state that.

4. REFACTORED CODE & EXPLANATION:
Provide an improved version of the code inside a code block.
Then briefly explain the important changes and why they improve the code.

Keep the review technically accurate, concise, and beginner-friendly.
"""

    # ========================================
    # 3. SEND CODE TO GEMINI
    # ========================================

    try:
        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )

        text = response.text

        # ====================================
        # 4. SPLIT GEMINI RESPONSE
        # ====================================

        bugs = (
            text.split("2. CODE STYLE")[0]
            .replace("1. BUG DETECTED:", "")
            .strip()
            if "1. BUG DETECTED:" in text
            else text
        )

        style = (
            text.split("3. SECURITY")[0]
            .split("2. CODE STYLE & PEP8:")[1]
            .strip()
            if "2. CODE STYLE & PEP8:" in text
            else "N/A"
        )

        security = (
            text.split("4. REFACTORED")[0]
            .split("3. SECURITY ANALYSIS:")[1]
            .strip()
            if "3. SECURITY ANALYSIS:" in text
            else "N/A"
        )

        refactored = (
            text.split("4. REFACTORED CODE & EXPLANATION:")[1]
            .strip()
            if "4. REFACTORED CODE & EXPLANATION:" in text
            else text
        )

        return (
            bugs,
            style,
            security,
            refactored
        )

    except Exception as e:
        error_message = f"Error during analysis: {str(e)}"
        return (
            error_message,
            error_message,
            error_message,
            error_message
        )


# ============================================
# 5. GRADIO INTERFACE
# ============================================

with gr.Blocks(title="AI Code Review Automation Agent") as app:

    gr.Markdown("# AI Code Review Automation Agent")

    gr.Markdown(
        "Submit code snippets to receive AI-powered bug analysis, "
        "code-quality reviews, security checks, and refactored code."
    )

    # ========================================
    # MAIN LAYOUT
    # ========================================

    with gr.Row():

        # ------------------------------------
        # LEFT SIDE - INPUT
        # ------------------------------------

        with gr.Column(scale=1):

            language_dropdown = gr.Dropdown(
                choices=[
                    "Python",
                    "JavaScript",
                    "C++",
                    "Java",
                    "Go",
                    "SQL"
                ],
                value="Python",
                label="Select Programming Language"
            )

            code_input = gr.Code(
                label="Source Code",
                lines=15,
                language="python"
            )

            analyze_btn = gr.Button(
                "Analyze & Review Code",
                variant="primary"
            )

        # ------------------------------------
        # RIGHT SIDE - OUTPUT
        # ------------------------------------

        with gr.Column(scale=1):

            with gr.Tabs():

                with gr.TabItem("Bugs & Errors"):
                    bugs_output = gr.Markdown()

                with gr.TabItem("Style & Standards"):
                    style_output = gr.Markdown()

                with gr.TabItem("Security Audit"):
                    security_output = gr.Markdown()

                with gr.TabItem("Refactored Code"):
                    refactored_output = gr.Markdown()

    # ========================================
    # BUTTON ACTION
    # ========================================

    analyze_btn.click(
        fn=review_code,
        inputs=[
            code_input,
            language_dropdown
        ],
        outputs=[
            bugs_output,
            style_output,
            security_output,
            refactored_output
        ]
    )


# ============================================
# 6. LAUNCH APPLICATION
# ============================================

if __name__ == "__main__":
    app.launch(
        theme=gr.themes.Soft(),
        share=True
    )

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bc8844f286ebcd36ab.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
